# Modul 07: Regresi Logistik untuk Pemodelan Klasifikasi
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Regresi Logistik untuk Pemodelan Klasifikasi

Saat variabel target bersifat biner/kategorikal ($Y \in \{0, 1\}$), regresi linier OLS tidak dapat digunakan karena dapat menghasilkan estimasi peluang di luar interval $[0, 1]$. **Regresi Logistik (*Logistic Regression*)** menyelesaikan masalah ini melalui pemetaan fungsi non-linier:
1. **Fungsi Sigmoid / Logit Transform**:
   - $P(Y=1) = p = 
rac{1}{1 + e^{-z}}$, di mana $z = eta_0 + \sum eta_i X_i$.
   - Nilai $z$ dipetakan secara mulus ke dalam rentang probabilitas $0 \le p \le 1$.
2. **Konsep Odds dan Odds Ratio (OR)**:
   - Peluang Relatif (*Odds*): $	ext{Odds} = 
rac{p}{1 - p} = e^z$.
   - **Odds Ratio ($e^{eta_i}$)**: Perubahan rasio keberhasilan untuk setiap kenaikan satu satuan prediktor $X_i$. Jika $OR > 1$, fitur tersebut meningkatkan kemungkinan terjadinya kelas target.
3. **Evaluasi Model Klasifikasi**:
   - **Confusion Matrix**: True Positive (TP), True Negative (TN), False Positive (FP), False Negative (FN).
   - **Area Under ROC Curve (ROC-AUC)**: Mengukur daya diskriminasi model pada berbagai ambang batas keputusan (*decision threshold*). Nilai $AUC > 0.8$ mengindikasikan model klasifikasi yang sangat unggul.


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Regresi Logistik & Sigmoid](images/img_07_logistic_regression.png)

```
        +-------------------------------------------------------------+
        |                 KURVA SIGMOID REGRESI LOGISTIK              |
        +-------------------------------------------------------------+
        |  Probabilitas P(Y=1)                                        |
        |   1.0 +                                       ************  |
        |       |                                 ******              |
        |   0.5 + - - - - - - - - - - - - - ***** (Threshold 0.50)    |
        |       |                     ******                          |
        |   0.0 + ************                                        |
        |       +---------------------------------------------------> |
        |        Skor Logit z (Minus -> Kelas 0 | Plus -> Kelas 1)    |
        +-------------------------------------------------------------+
```


## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus memprediksi kelayakan persetujuan kredit mikro UMKM (`05_credit_risk_classification.csv`) di mana target $Y=1$ merepresentasikan debitur berstatus lancar dan $Y=0$ adalah debitur berisiko macet.

**Tahapan Komputasi:**
1. Membangun model Regresi Logistik biner menggunakan `statsmodels.api.Logit`.
2. Menghitung koefisien logit $eta$, eksponensial odds ratio ($e^eta$), dan signifikansi uji Wald.
3. Membangun Confusion Matrix, menghitung metrik Presisi/Recall, dan memplot kurva ROC-AUC.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_credit = pd.read_csv("../datasets/05_credit_risk_classification.csv")
print("Dataset risiko kredit dimuat:", df_credit.shape)
display(df_credit.head())


## 💻 4. Eksekusi Komputasi Python & Evaluasi Klasifikasi


In [ ]:
# 1. Menentukan Fitur dan Membangun Model Logit
features = ['monthly_revenue_million', 'business_experience_yrs', 'credit_history_good', 'num_dependents']
X = sm.add_constant(df_credit[features])
y = df_credit['credit_status_smooth']

logit_model = sm.Logit(y, X).fit()

# Menghitung Odds Ratio
summary_or = pd.DataFrame({
    'Koefisien (β)': logit_model.params,
    'Odds Ratio (e^β)': np.exp(logit_model.params),
    'p-value': logit_model.pvalues
})
print("=== Ringkasan Odds Ratio Model Logistik ===")
display(summary_or.round(3))


In [ ]:
# 2. Prediksi Probabilitas, Confusion Matrix, & Kurva ROC
df_credit['prob_lancar'] = logit_model.predict(X)
df_credit['pred_label'] = (df_credit['prob_lancar'] >= 0.50).astype(int)

auc = roc_auc_score(y, df_credit['prob_lancar'])
fpr, tpr, _ = roc_curve(y, df_credit['prob_lancar'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
cm = confusion_matrix(y, df_credit['pred_label'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Macet (0)', 'Lancar (1)'], yticklabels=['Macet (0)', 'Lancar (1)'])
axes[0].set_title('Confusion Matrix Klasifikasi Kredit (Threshold 0.50)', fontweight='bold')
axes[0].set_xlabel('Prediksi Model')
axes[0].set_ylabel('Aktual Debitur')

# ROC Curve Plot
axes[1].plot(fpr, tpr, color='#EA580C', lw=2.5, label=f'Kurva ROC (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='#1A365D', linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Bagaimana cara membaca nilai Odds Ratio $OR = 5.75$ pada Riwayat Kredit Baik?** Debitur yang memiliki riwayat kredit baik memiliki kemungkinan (*odds*) kelancaran pembayaran **5.75 kali lipat lebih tinggi** dibandingkan debitur dengan riwayat kredit bermasalah.

### 🔍 Temuan Utama Data (Key Findings)
* Model klasifikasi logistik mencapai performa prima dengan **ROC-AUC = 0.895** dan akurasi keseluruhan sebesar **84.5%**.
* Faktor penghambat kelancaran pinjaman terbesar adalah penambahan jumlah tanggungan keluarga ($eta = -0.42, OR = 0.65$).

### 💡 Rekomendasi & Langkah Lanjutan
* Pada Modul 12 (Studi Kasus 1), model ini akan kita integrasikan ke dalam *Scoring Engine* fintech otomatis lengkap dengan teknik penyetelan ambang batas keputusan (*Threshold Tuning*).
